In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib, os
from pathlib import Path

project_root = Path(Path.cwd()).parent
file_path = project_root / 'data' / 'raw' / 'student_mental_health_burnout_1M.csv'
proc_dir = project_root / 'data' / 'processed'
model_dir = project_root / 'models'
os.makedirs(proc_dir,  exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

In [7]:

df = pd.read_csv(file_path)
print(f"Shape: {df.shape}")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\ml-final-coba\\ml-final-st\\branch-data\\raw\\student_mental_health_burnout_1M.csv'

In [ ]:

df["stress_category"] = pd.qcut(
    df["stress_level"],
    q=3,
    labels=["Low", "Moderate", "High"]
)

bin_edges = pd.qcut(df["stress_level"], q=3, retbins=True)[1]
joblib.dump(bin_edges, f"{model_dir}/bin_edges.pkl")

print("Target distribution:")
print(df["stress_category"].value_counts().sort_index())
print(f"\nBin edges: {bin_edges.round(4)}")

Target distribution:
stress_category
Low         333333
Moderate    333334
High        333333
Name: count, dtype: int64

Bin edges: [ 0.      3.5149  4.9743 10.    ]


In [ ]:
drop_cols = ["stress_level", "risk_level", "dropout_risk", "mental_health_index"]
df = df.drop(columns=drop_cols)
print(f"Kolom setelah drop: {list(df.columns)}")

Kolom setelah drop: ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure', 'academic_performance', 'anxiety_score', 'depression_score', 'sleep_hours', 'physical_activity', 'social_support', 'screen_time', 'internet_usage', 'financial_stress', 'family_expectation', 'burnout_score', 'mental_health_index', 'stress_category']


In [ ]:

print("Missing values sebelum handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = ["gender"]

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing values setelah handling:")
print(df.isnull().sum().sum(), "total missing")

Missing values sebelum handling:
Series([], dtype: int64)


C:\Users\zahra\AppData\Local\Temp\ipykernel_14896\3167203651.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\zahra\AppData\Local\Temp\ipykernel_14896\3167203651.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp


Missing values setelah handling:
0 total missing


In [ ]:

le_gender = LabelEncoder()
df["gender"] = le_gender.fit_transform(df["gender"])

mapping = dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))
print(f"Gender encoding: {mapping}")

joblib.dump(le_gender, f"{model_dir}/le_gender.pkl")

Gender encoding: {'Female': 0, 'Male': 1, 'Other': 2}


['../models/le_gender.pkl']

In [ ]:

target_mapping = {"Low": 0, "Moderate": 1, "High": 2}
df["target"] = df["stress_category"].map(target_mapping)

joblib.dump(target_mapping, f"{model_dir}/target_mapping.pkl")
print(f"Target mapping: {target_mapping}")
print(df["target"].value_counts().sort_index())

Target mapping: {'Low': 0, 'Moderate': 1, 'High': 2}
target
0    333333
1    333334
2    333333
Name: count, dtype: int64


In [ ]:

feature_cols = [c for c in df.columns if c not in ["stress_category", "target"]]
print(f"Jumlah fitur : {len(feature_cols)}")
print(f"Fitur : {feature_cols}")

X = df[feature_cols].values
y = df["target"].values

joblib.dump(feature_cols, f"{model_dir}/feature_cols.pkl")

Jumlah fitur : 17
Fitur : ['age', 'gender', 'academic_year', 'study_hours_per_day', 'exam_pressure', 'academic_performance', 'anxiety_score', 'depression_score', 'sleep_hours', 'physical_activity', 'social_support', 'screen_time', 'internet_usage', 'financial_stress', 'family_expectation', 'burnout_score', 'mental_health_index']


['../models/feature_cols.pkl']

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y         
)

print(f"Train : {X_train.shape[0]:,} rows")
print(f"Test : {X_test.shape[0]:,} rows")
print(f"\nTrain class dist : {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test  class dist : {dict(zip(*np.unique(y_test,  return_counts=True)))}")

Train : 800,000 rows
Test : 200,000 rows

Train class dist : {0: 266666, 1: 266667, 2: 266667}
Test  class dist : {0: 66667, 1: 66667, 2: 66666}


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Mean (5 fitur pertama) : {scaler.mean_[:5].round(3)}")
print(f"Std (5 fitur pertama) : {scaler.scale_[:5].round(3)}")

joblib.dump(scaler, f"{model_dir}/scaler2.pkl")

Mean (5 fitur pertama) : [22.994  0.561  2.501  5.002  5.999]
Std (5 fitur pertama) : [3.743 0.572 1.118 1.989 1.549]


['../models/scaler.pkl']

In [ ]:

np.save(f"{proc_dir}/X_train2.npy", X_train_scaled)
np.save(f"{proc_dir}/X_test2.npy",  X_test_scaled)
np.save(f"{proc_dir}/y_train2.npy", y_train)
np.save(f"{proc_dir}/y_test2.npy",  y_test)

for f in os.listdir(proc_dir):
    size = os.path.getsize(f"{proc_dir}/{f}") / 1e6
    print(f"{f} ({size:.1f} MB)")

X_test.npy (27.2 MB)
X_train.npy (108.8 MB)
y_test.npy (1.6 MB)
y_train.npy (6.4 MB)
